# Задание 1: кластеризация предложений Википедии по языкам

## 1. Импорты, проверка и форматирование

In [1]:
%pip install -q wikipedia requests scikit-learn scipy seaborn matplotlib pandas numpy sentence-transformers fasttext tqdm

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import re
import time
import json
import warnings

from collections import Counter

import numpy as np
import pandas as pd
import requests
import wikipedia
from tqdm.auto import tqdm
import seaborn as sns
import matplotlib.pyplot as plt
from IPython.display import display

%matplotlib inline
warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 120)

SEED = 42

YOUR_EMAIL = 'oyo.alena.oyo@gmail.com'
wikipedia.set_user_agent(f'HSE-NLP-course/0.1 ({YOUR_EMAIL})')

In [3]:
def check(condition, message):
    # проверка, работает ли код, чтобы ловить баги: ✓ - да, ✕ - нет
    print(('✓ ' if condition else '✕ ') + message)
    return condition

In [35]:
# Оформление таблиц (как в прошлой домашке)
def show_table(df, formats=None):
    styler = (
        df.style
        .hide(axis="index")
        .set_properties(
            subset=[df.columns[0]],
            **{
                "text-align": "left",
                "font-weight": "bold",
            }
        )
    )

    if len(df.columns) > 1:
        styler = styler.set_properties(
            subset=list(df.columns[1:]),
            **{"text-align": "center"}
        )

    styler = styler.set_table_styles([
        {
            "selector": "th",
            "props": [("text-align", "center")]
        },
        {
            "selector": "th.col0",
            "props": [
                ("text-align", "left"),
                ("font-weight", "bold"),
            ]
        },
    ])

    if formats:
        styler = styler.format(formats)

    display(styler)

## 2. Выбор языков, скачивание и предобработка

In [93]:
MIN_INTERVAL = 0.35
_last_call = [0.0]


def _wiki_request(params):
    params['format'] = 'json'
    params.setdefault('action', 'query')
    for attempt in range(5):
        wait = MIN_INTERVAL - (time.time() - _last_call[0])
        if wait > 0:
            time.sleep(wait)
        _last_call[0] = time.time()
        r = requests.get(wikipedia.wikipedia.API_URL, params=params,
                         headers={'User-Agent': wikipedia.wikipedia.USER_AGENT}, timeout=30)
        if r.status_code == 429: # Too Many Requests - ждем, сколько просит сервер
            retry_after = int(r.headers.get('Retry-After', 30))
            print(f'429 Too Many Requests, ждем {retry_after} с')
            time.sleep(retry_after)
            continue
        r.raise_for_status()
        return r.json()
    r.raise_for_status()

wikipedia.wikipedia._wiki_request = _wiki_request

**Выбор языков:** две пары близких языков и один «одиночка»: датский (`da`) и норвежский (`no`), индонезийский (`id`) и малайский. Близкородственные пары берём специально: их кластеризаторы путают чаще всего, так что в разборе ошибок будет на что посмотреть.

In [37]:
code2lang = wikipedia.languages()

langs = ['da', 'no', 'id', 'ms', 'fi']

for lang in langs:
    print(f'{lang} - {code2lang[lang]}')

N_ARTICLES = 10
MIN_ARTICLE_CHARS = 2000
MIN_SENTENCES_PER_ARTICLE = 20
MAX_SENTENCES_PER_ARTICLE = 40
RANDOM_STATE = 42

NON_SENTENCE = re.compile(
    r'^\(?\d{4}[,).]'  # (2022, ... / 1978)
    r'|^[A-ZÅÆØÄÖÜ][^,]{1,40},\s*[A-Z]'  # Hederberg, Hans
    r'|\b(?:Press|Forlag|Publishing)\.?$' # Oxford University Press
    r'|\b(?:et al\.|toim\.)$'  # et al. / toim.
    r'|\b(?:r\.os\.|r\.y\.|os\.)$' # финские списки организаций
    ,
    flags=re.I
)

# предобработка

def clean_article(text):
    text = text.replace('\xa0', ' ').replace('\u200b', '').replace('\xad', '')
    text = re.sub(r'^\s*==+.*?==+\s*$', '', text, flags=re.M)
    text = re.sub(r'https?://\S+', ' ', text)
    text = re.sub(r'\[\d+\]', '', text)
    text = re.sub(r'\blähde\?', '', text, flags=re.I)
    lines = [re.sub(r'[ \t]+', ' ', line).strip() for line in text.splitlines()]
    return [line for line in lines if line]

SENT_FINAL = re.compile(r'[.!?…]["»”’)\]]*$')
TECH_LINE = re.compile(r'\b(?:cm³|kW|hk|Nm|kg|liter|cyl\.|omdr\./min)\b', flags=re.I)

def is_prose_line(line):
    words = len(line.split())
    if words <= 2:
        return False
    if words < 5 and not SENT_FINAL.search(line):
        return False
    if words < 10 and TECH_LINE.search(line):
        return False
    return True

ABBREVIATIONS = {'s', 'k', 'mm', 'vs'}
SENT_END = re.compile(r'[.!?…]+["»”’)\]]*\s+')

def is_false_boundary(prev, nxt):
    if not prev or not nxt:
        return False
    last = prev.split()[-1]
    if nxt[0].islower():
        return True
    token = last.strip('(').rstrip('.').lower()
    if token in ABBREVIATIONS:
        return True
    return bool(re.fullmatch(r'[^\W\d_]\.', last)) and last[0].isupper()

def split_sentences(paragraph):
    pieces, start = [], 0
    for m in SENT_END.finditer(paragraph):
        pieces.append(paragraph[start:m.end()].strip())
        start = m.end()
    pieces.append(paragraph[start:].strip())

    sentences = []
    for piece in pieces:
        if not piece:
            continue
        if sentences and is_false_boundary(sentences[-1], piece):
            sentences[-1] += ' ' + piece
        else:
            sentences.append(piece)
    return sentences

def normalize_sentence(sentence):
    return re.sub(r'\.{2,}$', '.', sentence.strip())

BIBLIO_LINE = re.compile(
    r'ISBN|ISSN|doi:|https?://|www\.'
    r'|^\(?\d{4},[^)]*\)\.?$'  # (2022, Mei 5).
    r'|^[^.!?]{0,80}\(\d{4}\)\.?$'  # Hederberg, Hans (1978).
    r'|^[A-ZÅÆØÄÖÜ][^:]{0,40}:\s+[^,]{1,50},\s*\d{4}\.?$'  # Helsinki: Tammi, 2016.
    r'|\b(?:University Press|Press|Publishing|Forlag|Verlag)\.?$'
    r'|\bet al\.?\)?$'
    r'|\btoim\.?$',
    flags=re.I
)

def is_good_sentence(sentence):
    if len(sentence.split()) <= 2:
        return False
    if BIBLIO_LINE.search(sentence):
        return False
    if NON_SENTENCE.search(sentence):
        return False
    if not SENT_FINAL.search(sentence):
        return False
    return True

def get_good_sentences(text):
    sentences = []
    for line in clean_article(text):
        if not is_prose_line(line):
            continue
        for sentence in split_sentences(line):
            sentence = normalize_sentence(sentence)
            if is_good_sentence(sentence):
                sentences.append(sentence)
    return list(dict.fromkeys(sentences))

da - dansk
no - norsk
id - Bahasa Indonesia
ms - Bahasa Melayu
fi - suomi


In [38]:
# загрузка статей

def load_with_disambigution(page):
    try:
        p = wikipedia.page(page, auto_suggest=False)
    except wikipedia.DisambiguationError as e:
        random_option = np.random.choice(e.options)
        p = wikipedia.page(random_option, auto_suggest=False)
    return p

def get_texts_for_lang(lang, existing_articles=None, n=N_ARTICLES,
                       min_chars=MIN_ARTICLE_CHARS, min_sentences=MIN_SENTENCES_PER_ARTICLE,
                       max_batches=20):
    wikipedia.set_lang(lang)
    existing_articles = existing_articles or []
    articles, skipped = [], Counter()
    seen_titles = {a['title'] for a in existing_articles}
    start = time.perf_counter()

    # сначала проверяем старый кэш
    for art in existing_articles:
        if len(art['text']) < min_chars:
            skipped['кэш: слишком короткая'] += 1
            continue
        n_sent = len(get_good_sentences(art['text']))
        if n_sent < min_sentences:
            skipped[f'кэш: < {min_sentences} предложений'] += 1
            continue
        articles.append({'title': art['title'], 'text': art['text']})
        if len(articles) >= n:
            break

    print(f'[{lang}] из кэша подходит {len(articles)} из {len(existing_articles)}')

    
    with tqdm(total=n, initial=len(articles), desc=f'[{lang}] статьи', unit='ст.') as bar:
        for _ in range(max_batches):
            if len(articles) >= n:
                break

            for page_name in wikipedia.random(10):
                if len(articles) >= n:
                    break
                if page_name in seen_titles:
                    skipped['повтор названия'] += 1
                    continue
                seen_titles.add(page_name)
                bar.set_postfix_str(page_name[:40])

                try:
                    page = load_with_disambigution(page_name)
                except Exception as e:
                    skipped['ошибка загрузки'] += 1
                    tqdm.write(f'  [{lang}] пропуск «{page_name}»: {(str(e).strip().splitlines() or [""])[0][:90]}')
                    continue

                if page.title != page_name and page.title in seen_titles:
                    skipped['повтор названия'] += 1
                    continue
                seen_titles.add(page.title)

                if len(page.content) < min_chars:
                    skipped[f'короче {min_chars} симв.'] += 1
                    continue

                n_sent = len(get_good_sentences(page.content))
                if n_sent < min_sentences:
                    skipped[f'< {min_sentences} предложений'] += 1
                    continue

                articles.append({'title': page.title, 'text': page.content})
                bar.update(1)

    mean_len = np.mean([len(a['text']) for a in articles]) if articles else 0
    print(f'[{lang}] получено {len(articles)} из {n} за {time.perf_counter() - start:.0f} с, '
          f'средняя длина {mean_len:.0f} симв.; пропущено: {dict(skipped) or "ничего"}')
    return articles[:n]

In [39]:
# кэш: проверяем старые статьи и при необходимости добираем новые

WIKI_CACHE = 'hw_wiki_articles_' + '_'.join(langs) + '.json'

print(f'Языки: {langs}; по {N_ARTICLES} статей; минимум {MIN_SENTENCES_PER_ARTICLE} предложений; кэш: {WIKI_CACHE}')

if os.path.exists(WIKI_CACHE):
    wiki_articles = json.load(open(WIKI_CACHE, encoding='utf-8'))
    print(f'Нашёлся кэш: {sum(len(a) for a in wiki_articles.values())} статей')
else:
    wiki_articles = {}

t0 = time.perf_counter()

for lang in langs:
    old_articles = wiki_articles.get(lang, [])
    try:
        wiki_articles[lang] = get_texts_for_lang(lang, old_articles, N_ARTICLES)
    except Exception as e:
        print('ERROR ON -', lang, str(e).strip('\n'))
        continue

    with open(WIKI_CACHE, 'w', encoding='utf-8') as f:
        json.dump(wiki_articles, f, ensure_ascii=False)
    print(f'[{lang}] сохранено в {WIKI_CACHE}\n')
    time.sleep(1)

print(f'Всего: {time.perf_counter() - t0:.0f} с')

check(
    all(len(wiki_articles.get(lang, [])) == N_ARTICLES for lang in langs),
    f'для каждого из {len(langs)} языков есть по {N_ARTICLES} подходящих статей'
)

Языки: ['da', 'no', 'id', 'ms', 'fi']; по 10 статей; минимум 20 предложений; кэш: hw_wiki_articles_da_no_id_ms_fi.json
Нашёлся кэш: 50 статей
[da] из кэша подходит 10 из 10


[da] статьи: 100%|##########| 10/10 [00:00<?, ?ст./s]

[da] получено 10 из 10 за 0 с, средняя длина 8100 симв.; пропущено: ничего
[da] сохранено в hw_wiki_articles_da_no_id_ms_fi.json

[no] из кэша подходит 10 из 10


[no] статьи: 100%|##########| 10/10 [00:00<?, ?ст./s]

[no] получено 10 из 10 за 0 с, средняя длина 4773 симв.; пропущено: ничего
[no] сохранено в hw_wiki_articles_da_no_id_ms_fi.json

[id] из кэша подходит 10 из 10


...

[ms] статьи: 100%|##########| 10/10 [00:00<?, ?ст./s]

[ms] получено 10 из 10 за 0 с, средняя длина 8937 симв.; пропущено: ничего
[ms] сохранено в hw_wiki_articles_da_no_id_ms_fi.json

[fi] из кэша подходит 10 из 10


[fi] статьи: 100%|##########| 10/10 [00:00<?, ?ст./s]

[fi] получено 10 из 10 за 0 с, средняя длина 6633 симв.; пропущено: ничего
[fi] сохранено в hw_wiki_articles_da_no_id_ms_fi.json

Всего: 5 с
✓ для каждого из 5 языков есть по 10 подходящих статей


True

In [40]:
# проверка: сколько предложений получилось в каждой статье

article_stats = []
for lang in langs:
    for art_id, art in enumerate(wiki_articles[lang]):
        n_sent = len(get_good_sentences(art['text']))
        article_stats.append({
            'lang': lang,
            'article_id': f'{lang}_{art_id}',
            'title': art['title'],
            'n_sentences': n_sent,
            'used': min(n_sent, MAX_SENTENCES_PER_ARTICLE)
        })

article_stats = pd.DataFrame(article_stats)
display(article_stats)

display(
    article_stats.groupby('lang').agg(
        articles=('article_id', 'count'),
        min=('n_sentences', 'min'),
        mean=('n_sentences', 'mean'),
        max=('n_sentences', 'max'),
        used=('used', 'sum')
    ).reindex(langs)
)

,lang,article_id,title,n_sentences,used
0,da,da_0,Toyota Corolla,128,40
1,da,da_1,Samnitter,21,21
2,da,da_2,Hendrick Cornelius Vroom,20,20
3,da,da_3,Operation Leo,22,22
4,da,da_4,Leopold 1. af Belgien,41,40
5,da,da_5,Christian Møller (fysiker),20,20
6,da,da_6,Dennis Rommedahl,21,21
7,da,da_7,Kalsoy,48,40
8,da,da_8,Slaget ved Rafah,134,40
9,da,da_9,Hans Talhoffer,28,28


,articles,min,mean,max,used
lang,,,,,
da,10,20,48.3,134,292
no,10,20,32.7,72,294
id,10,23,86.9,379,326
ms,10,20,43.4,122,319
fi,10,20,47.5,125,309


## 3. Проверка предобработки, промежуточный кэш и сохранение

In [41]:
audit = sent_df.groupby('lang', group_keys=False).sample(n=50, random_state=42)
display(audit[['lang', 'title', 'sentence']])

,lang,title,sentence
206,da,Christian Møller (fysiker),Han blev Ridder af Dannebrog 1952 og Ridder af 1. grad 1960.
33,da,Toyota Corolla,"Motorerne blev let modificeret, og opfyldt nu Euro4-normen, og en ny økonomisk dieselmotor på 1,4 liter med 66 kW (9..."
163,da,Leopold 1. af Belgien,Som det eneste legitime barn af tronfølgeren havde hun udsigt til at blive regerende dronning efter sin far og søgte...
78,da,Samnitter,Det var sjældent – skønt muligt – for samniterne at forene sig under en koalition; normalt fungerede stammerne og by...
93,da,Hendrick Cornelius Vroom,"Bogen er en eventyrhistorie og indeholder bl.a. beskrivelser af, at Vrooms bukser frøs fast til en bjergtop, og at h..."
...,...,...,...
385,no,Idiocracy,"Men de minst intelligente menneskene re-produserer seg veldig ofte, noe som skaper stadig dummere generasjoner."
492,no,Rosh HaNikra grenseovergang,I 2007 ble liket av Gabriel DeWitt overført til Israel ved krysset.
575,no,Marie Antoinette (film fra 2006),Rip Torn som Louis XV av Frankrike.
572,no,Marie Antoinette (film fra 2006),"Da Marie Antoinette til slutt begynner å finne sin plass i livet, blir hun dratt inn i en blodig revolusjon, tatt ti..."


In [42]:
tmp = sent_df.assign(
    words=sent_df.sentence.str.split().str.len(),
    chars=sent_df.sentence.str.len()
)

display(tmp.sort_values('words')[['lang', 'title', 'sentence', 'words']].head(100))
display(tmp.sort_values('chars', ascending=False)[['lang', 'title', 'sentence', 'chars']].head(50))

,lang,title,sentence,words
1539,fi,Suomen Työväen Raittiusliitto,Vallilan työväen r.y.,3
649,id,Vithoba,The Haridasa movement.,3
924,id,Bahasa Inggris India,"K. Paul, Trench.",3
701,id,"Putabangun, Bontoharu, Kepulauan Selayar",Lalaki Binongko Dg.,3
1538,fi,Suomen Työväen Raittiusliitto,Turun nuorten r.os.,3
...,...,...,...,...
1485,fi,Rutto,Ruttobakteerista voi kehittyä lääkeresistentti kanta.,5
1482,fi,Rutto,Rutto riehuu etenkin lämpiminä vuodenaikoina.,5
1481,fi,Rutto,Ruttokirput pystyvät elämään myös viljalla.,5
1479,fi,Rutto,Alettuaan oireet kestävät 3–5 vuorokautta.,5


,lang,title,sentence,chars
716,id,"Putabangun, Bontoharu, Kepulauan Selayar",Setelah sepeninggalan Tenri Dio (Opu pertama Putabangun) gong besar ini menjadi gaukang kerajaan putabangun turun te...,711
821,id,Jaringan telekomunikasi,"Jika dalam jaringan tidak ada aktivitas atau bersih komputer akan mentransmisikan data, jika ada transmisi lain di d...",521
859,id,Hurtcore,"Korban Schellenberger yang berusia enam tahun (yang juga pernah ia upayakan untuk dibunuh, bersama ibu anak tersebut...",491
866,id,Hurtcore,Video-video tersebut menampilkan salah satu pacar Filipina Scully (Liezyl Margallo Castaña) yang menyiksa balita ter...,477
873,id,Hurtcore,William Michael Spearman – pria asal Alabama yang selama lima tahun mengoperasikan sebuah situs web pornografi anak ...,465
1136,ms,Mahkamah Agung Bangladesh,"Majlis kehakiman tertinggi ditubuhkan dengan Ketua Hakim Bangladesh dan dua hakim kanan Bahagian Rayuan seterusnya, ...",449
1155,ms,Mahkamah Agung Bangladesh,Presiden Bangladesh memerintahkan penubuhan Majlis Kehakiman Tertinggi untuk menyiasat dakwaan salah laku hakim Mahk...,407
693,id,"Putabangun, Bontoharu, Kepulauan Selayar",Pada masa setelah We Tenri Dio seorang Raja Putabangung bergelar Lalaki merujuk pada gelar kebangsawanan suami We Te...,403
673,id,Hukum di Vatikan,"Pada tanggal 7 Mei 2015, Paus Fransiskus diangkat sebagai Hakim Pengadilan Gerejawi Negara Kota Vatikan, Lucio Baner...",386
1138,ms,Mahkamah Agung Bangladesh,"Mengikut Perkara 111 Perlembagaan Bangladesh, 1972, penghakiman Mahkamah Agung mempunyai kesan yang mengikat dan art...",385


In [43]:
display(
    tmp.groupby('lang').agg(
        sentences=('sentence', 'count'),
        mean_words=('words', 'mean'),
        median_words=('words', 'median'),
        min_words=('words', 'min'),
        max_words=('words', 'max'),
        mean_chars=('chars', 'mean')
    ).reindex(langs).round(1)
)

,sentences,mean_words,median_words,min_words,max_words,mean_chars
lang,,,,,,
da,315,18.5,16.0,3,62,117.2
no,295,16.9,16.0,3,53,105.3
id,341,18.9,16.0,3,90,137.1
ms,349,16.7,15.0,3,69,121.1
fi,331,10.6,10.0,3,33,91.5


In [44]:
tmp = sent_df.assign(words=sent_df.sentence.str.split().str.len(), chars=sent_df.sentence.str.len())

short = tmp[tmp.words <= 4]
print('Коротких:', len(short), f'({len(short)/len(tmp):.1%})')
with pd.option_context('display.max_rows', None):
    display(short[['lang', 'title', 'sentence', 'words']])

Коротких: 82 (5.0%)


,lang,title,sentence,words
87,da,Hendrick Cornelius Vroom,Hendrik Cornelisz Vroom (ca.,4
148,da,Operation Leo,"Hederberg, Hans (1978).",3
149,da,Operation Leo,Sveriges Radio P3 Dokumentär.,4
205,da,Christian Møller (fysiker),Ørsted Medaljen 1970.,3
273,da,Kalsoy,"Føroya Skúlabókagrunnur, 1999.",3
274,da,Kalsoy,"Politikens Forlag, 2005.",3
396,no,Idiocracy,Maya Rudolph som Rita.,4
453,no,Slaget ved Kalisz,"Frost, Robert I (2000).",4
454,no,Slaget ved Kalisz,The Northern Wars.,3
456,no,Slaget ved Kalisz,"""Der Ausführlichen Lebens-Beschreibung.",3


In [50]:
#промежуточный кэш

WIKI_CACHE = 'hw_wiki_articles_' + '_'.join(langs) + '.json'

with open(WIKI_CACHE, 'w', encoding='utf-8') as f:
    json.dump(wiki_articles, f, ensure_ascii=False)

print(f'Сохранено {sum(len(a) for a in wiki_articles.values())} статей в {WIKI_CACHE}')

Сохранено 50 статей в hw_wiki_articles_da_no_id_ms_fi.json


In [51]:
rows = []
rng = np.random.default_rng(RANDOM_STATE)

for lang in langs:
    for art_id, art in enumerate(wiki_articles[lang]):
        sentences = get_good_sentences(art['text'])

        if len(sentences) > MAX_SENTENCES_PER_ARTICLE:
            idx = np.sort(rng.choice(len(sentences), MAX_SENTENCES_PER_ARTICLE, replace=False))
            sentences = [sentences[i] for i in idx]

        for sent in sentences:
            rows.append({
                'lang': lang,
                'article_id': f'{lang}_{art_id}',
                'title': art['title'],
                'sentence': sent
            })

sent_df = pd.DataFrame(rows).drop_duplicates(subset=['lang', 'sentence']).reset_index(drop=True)

SENT_CSV = 'hw_sentences_' + '_'.join(langs) + '.csv'
sent_df.to_csv(SENT_CSV, index=False)
print(f'Сохранено {len(sent_df)} предложений в {SENT_CSV}')

Сохранено 1540 предложений в hw_sentences_da_no_id_ms_fi.csv


## 4. Эмбеддинги
Для каждого предложения получаем векторное представление с помощью мультиязычной модели `intfloat/multilingual-e5-base`. Для всех текстов используется префикс `query`.

In [54]:
from sentence_transformers import SentenceTransformer

MODEL_NAME = 'intfloat/multilingual-e5-base'
print(f'Загружаем {MODEL_NAME} ...')
model = SentenceTransformer(MODEL_NAME)
print('Устройство:', model.device)

start = time.perf_counter()
X = model.encode(
    ['query: ' + s for s in data['sentence']],
    batch_size=32,
    show_progress_bar=True,
    normalize_embeddings=True,
    convert_to_numpy=True,
)
print(f'Готово: {X.shape[0]} векторов размерности {X.shape[1]} за {time.perf_counter() - start:.1f} с')

check(X.shape[0] == len(data), 'число векторов = числу предложений')
check(not np.isnan(X).any(), 'в эмбеддингах нет NaN')
check(np.allclose(np.linalg.norm(X, axis=1), 1, atol=1e-3), 'векторы нормированы (длина 1)')

# насколько вообще эмбеддинги «видят» язык: средняя косинусная близость внутри языка и между языками
S = X @ X.T
same_lang = y_true[:, None] == y_true[None, :]
print(f'средняя косинусная близость: внутри одного языка {S[same_lang & ~np.eye(len(X), dtype=bool)].mean():.3f}, '
      f'между разными {S[~same_lang].mean():.3f}')

Загружаем intfloat/multilingual-e5-base ...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Устройство: mps:0


Batches:   0%|          | 0/46 [00:00<?, ?it/s]

Готово: 1460 векторов размерности 768 за 18.5 с
✓ число векторов = числу предложений
✓ в эмбеддингах нет NaN
✓ векторы нормированы (длина 1)
средняя косинусная близость: внутри одного языка 0.796, между разными 0.748


## 5. Кластеризация и ARI
Для кластеризации используется алгоритм KMeans. Число кластеров равно числу языков в корпусе — 5. Алгоритм не получает информацию об истинном языке предложений и группирует их только на основании близости эмбеддингов.

Качество кластеризации оценивается метрикой ARI, которая сравнивает полученные кластеры с истинными языковыми метками. ARI не зависит от нумерации кластеров, поэтому отдельно сопоставлять номера кластеров с языками для вычисления метрики не нужно.

In [81]:
data = sent_df.copy().reset_index(drop=True)
y_true = data['lang'].to_numpy()

print(data['lang'].value_counts().reindex(langs))

lang
da    292
no    294
id    326
ms    319
fi    309
Name: count, dtype: int64


In [71]:
from sentence_transformers import SentenceTransformer

MODEL_NAME = 'intfloat/multilingual-e5-base'
model = SentenceTransformer(MODEL_NAME)

X = model.encode(
    ['query: ' + s for s in data['sentence']],
    batch_size=32,
    show_progress_bar=True,
    normalize_embeddings=True,
    convert_to_numpy=True
)

print('Размер эмбеддингов:', X.shape)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/49 [00:00<?, ?it/s]

Размер эмбеддингов: (1540, 768)


In [73]:
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score

kmeans = KMeans(n_clusters=len(langs), n_init=10, random_state=RANDOM_STATE)
data['cluster'] = kmeans.fit_predict(X)

ari = adjusted_rand_score(y_true, data['cluster'])
print(f'ARI = {ari:.3f}')

ARI = 0.620


In [74]:
ct = pd.crosstab(data['lang'], data['cluster']).reindex(langs)
display(ct)

cluster,0,1,2,3,4
lang,,,,,
da,0,274,0,1,17
no,0,268,0,0,26
id,260,0,0,12,54
ms,19,0,0,235,65
fi,0,0,284,0,25


In [75]:
cluster2lang = ct.idxmax(axis=0).to_dict()
print('Основной язык каждого кластера:', cluster2lang)

Основной язык каждого кластера: {0: 'id', 1: 'da', 2: 'fi', 3: 'ms', 4: 'ms'}


In [76]:
data['cluster_lang'] = data['cluster'].map(cluster2lang)
errors = data[data['lang'] != data['cluster_lang']]

print('Ошибочно кластеризовано:', len(errors))

Ошибочно кластеризовано: 422


In [77]:
errors.groupby(['lang', 'cluster_lang']).size().sort_values(ascending=False)

lang  cluster_lang
no    da              268
id    ms               66
no    ms               26
fi    ms               25
ms    id               19
da    ms               18
dtype: int64

В результате кластеризации получено значение **ARI = 0.620**, что говорит о достаточно хорошем, но не полном совпадении кластеров с реальным разделением текстов по языкам. Основные ошибки связаны с близкородственными языками - датским/норвежским и индонезийским/малайским, которые оказываются близкими друг к другу в пространстве эмбеддингов.

## 5. Разбор ошибок

In [89]:
pairs = [('no', 'da'), ('ms', 'id'), ('id', 'ms')]

examples = pd.concat([
    errors[(errors['lang'] == true) & (errors['cluster_lang'] == cluster_lang)]
    .sample(1, random_state=RANDOM_STATE)
    for true, cluster_lang in pairs
])

display(examples[['lang', 'cluster_lang', 'title', 'sentence']])

,lang,cluster_lang,title,sentence
415,no,da,Rosh HaNikra grenseovergang,I tilfelle et britisk nederlag kunne tyskerne lett marsjere inn i Palestina og slå seg sammen med Vichyfranske styrk...
938,ms,id,Mak Dara,MAK DARA meninggal dunia pada 16 Februari 1970 akibat sakit tua di Hospital Besar Singapura.
812,id,ms,Hurtcore,"Matthew David Graham – pria Australia yang, dengan nama layar ""Lux"", mengoperasikan sebuah imperium situs web hurtco..."


In [90]:
for i, (_, row) in enumerate(examples.iterrows(), 1):
    print(f'Ошибка {i}')
    print(f'Истинный язык: {row["lang"]}')
    print(f'Основной язык кластера: {row["cluster_lang"]}')
    print(f'Статья: {row["title"]}')
    print(f'Текст: {row["sentence"]}\n')

Ошибка 1
Истинный язык: no
Основной язык кластера: da
Статья: Rosh HaNikra grenseovergang
Текст: I tilfelle et britisk nederlag kunne tyskerne lett marsjere inn i Palestina og slå seg sammen med Vichyfranske styrker i Libanon og Syria.

Ошибка 2
Истинный язык: ms
Основной язык кластера: id
Статья: Mak Dara
Текст: MAK DARA meninggal dunia pada 16 Februari 1970 akibat sakit tua di Hospital Besar Singapura.

Ошибка 3
Истинный язык: id
Основной язык кластера: ms
Статья: Hurtcore
Текст: Matthew David Graham – pria Australia yang, dengan nama layar "Lux", mengoperasikan sebuah imperium situs web hurtcore di web gelap dari kamar tidurnya di rumah orang tuanya di pinggiran Melbourne, South Morang, ketika ia berusia antara 19 hingga 21 tahun.



Основные ошибки кластеризации возникают между близкородственными языками. Наиболее заметно смешиваются датский и норвежский: значительная часть норвежских предложений попадает в кластер, где преобладают датские тексты. Это связано с высокой лексической и грамматической близостью этих языков.

Похожая ситуация наблюдается для малайского и индонезийского: встречаются ошибки `ms → id` и `id → ms`. Это показывает, что эмбеддинги хорошо отделяют более различающиеся языки, но хуже различают близкородственные. Таким образом, основные ошибки KMeans объясняются прежде всего сходством самих языков, а не случайным распределением текстов по кластерам.

# Задание 2: SynTagRus

## Подготовка

In [122]:
# !pip install conllu spacy

In [102]:
import conllu
import spacy
from collections import Counter
from spacy.tokens import Doc
from spacy.matcher import DependencyMatcher

nlp_blank = spacy.blank('ru')

def conllu_to_doc(sentence):
    words = [t for t in sentence if isinstance(t['id'], int)]
    if not words:
        return None
    position = {t['id']: k for k, t in enumerate(words)}
    return Doc(
        nlp_blank.vocab,
        words=[t['form'] for t in words],
        spaces=[(t['misc'] or {}).get('SpaceAfter') != 'No' for t in words],
        heads=[position.get(t['head'], k) for k, t in enumerate(words)],
        deps=['ROOT' if t['head'] == 0 else t['deprel'] for t in words],
        pos=[t['upos'] for t in words],
        lemmas=[t['lemma'] for t in words],
        morphs=['|'.join(f'{k}={v}' for k, v in (t['feats'] or {}).items()) for t in words],
    )

with open('ru_syntagrus-ud-dev.conllu', encoding='utf8') as f:
    syn_docs = [d for s in conllu.parse(f.read()) if (d := conllu_to_doc(s)) is not None]
len(syn_docs)

8906

## query 1: роль слова «который» в относительном придаточном
Ищем существительное с относительным придаточным (acl:relcl), где слово «который» напрямую зависит от глагола этого придаточного. По типу этой зависимости можно понять, какую роль «который» играет в придаточном: подлежащее («мастер, который дал»), дополнение («надежды, которые связывали») или обстоятельство («море, о котором говорил»). Сначала запрос находит и случаи, где вершина придаточного не глагол, поэтому добавляем фильтр и оставляем только придаточные с глаголом в вершине.

In [116]:
relcl_matcher = DependencyMatcher(nlp_blank.vocab)
relcl_matcher.add("RELCL", [[
    {"RIGHT_ID": "noun", "RIGHT_ATTRS": {"POS": {"IN": ["NOUN", "PROPN"]}}},
    # фильтр: вершина придаточного - глагол
    {"LEFT_ID": "noun", "REL_OP": ">", "RIGHT_ID": "verb",
     "RIGHT_ATTRS": {"DEP": "acl:relcl", "POS": "VERB"}},
    {"LEFT_ID": "verb", "REL_OP": ">", "RIGHT_ID": "rel", "RIGHT_ATTRS": {"LEMMA": "который"}},
]])

relcl_rows = []
for doc in syn_docs:
    for _, ids in relcl_matcher(doc):
        noun, verb, rel = ids
        marked = ' '.join(f'[{t.text}]' if t.i in ids else t.text for t in doc)
        relcl_rows.append((doc[rel].dep_, doc[noun].text, doc[verb].text, marked))

print(len(relcl_rows))
Counter(r[0] for r in relcl_rows).most_common()

424


[('nsubj', 205),
 ('obl', 124),
 ('obj', 50),
 ('nsubj:pass', 31),
 ('iobj', 8),
 ('obl:tmod', 6)]

In [117]:
for role in ['nsubj', 'obj', 'obl']:
    print(role)
    for dep, n, v, text in sorted((r for r in relcl_rows if r[0] == role), key=lambda r: len(r[3]))[:3]:
        print(f'  {n} <- {v}: {text}')

nsubj
  человек <- видит: Что означает [человек] , [который] НЕ [видит] ?
  человек <- примет: Над нами должен стоять [человек] , [который] [примет] решение .
  событием <- просветило: Рождение Платона было [событием] , [которое] [просветило] меня .
obj
  настоящее <- променяю: Я возвращаюсь в свое [настоящее] , [которое] ни на что не [променяю] .
  заседаниях <- проводил: Я присутствовал на многих [заседаниях] , [которые] [проводил] Королев .
  гипотеза <- выдвинули: Сегодня очень популярна [гипотеза] , [которую] [выдвинули] американцы .
obl
  платформу <- сдуло: Он увидел [платформу] , с [которой] словно [сдуло] людей .
  старик <- помянули: Тут и объявляется [старик] пьяница , о [котором] мы [помянули] вскользь .
  опухоли <- едешь: Лава заключена в земляные [опухоли] , по [которым] [едешь] , как по маслу .


## query 2: глаголы с зависимым инфинитивом (xcomp)
Ищем пары «глагол + инфинитив» (мочь делать, начать говорить, хотеть вернуться). Затем смотрим, какие глаголы чаще всего встречаются в таких конструкциях. Среди самых частых оказываются мочь, начать, стать, хотеть, позволять, пытаться и т.д. Такой результат можно использовать, чтобы находить глаголы, которые часто сочетаются с инфинитивом через `xcomp`. Сначала в результат попадают лишние случаи: деепричастия («ложась спать») и повторы одного и того же глагола («работать он не работал»), поэтому добавляем два фильтра.

In [118]:
inf_matcher = DependencyMatcher(nlp_blank.vocab)
inf_matcher.add("VERB_INF", [[
    {"RIGHT_ID": "head", "RIGHT_ATTRS": {"POS": "VERB"}},
    {"LEFT_ID": "head", "REL_OP": ">", "RIGHT_ID": "inf",
     "RIGHT_ATTRS": {"DEP": "xcomp", "POS": "VERB", "MORPH": {"IS_SUPERSET": ["VerbForm=Inf"]}}},
]])

inf_rows = []
for doc in syn_docs:
    for _, ids in inf_matcher(doc):
        h, i = doc[ids[0]], doc[ids[1]]
        if h.morph.get('VerbForm') == ['Conv'] or h.lemma_ == i.lemma_:
            continue
        marked = ' '.join(f'[{t.text}]' if t.i in ids else t.text for t in doc)
        inf_rows.append((h.lemma_, i.lemma_, marked))

print(len(inf_rows))

1026


In [119]:
top = Counter(h for h, _, _ in inf_rows).most_common(10)

for head, count in top:
    example = min((r for r in inf_rows if r[0] == head), key=lambda r: len(r[2]))
    print(f'{head} ({count}): {example[2]}')

мочь (302): Не [может] [быть] …
начать (55): Мы [начали] [выздоравливать] .
стать (46): Я [стал] [наблюдать] .
хотеть (43): Что [хочет] [показать] Медведев ?
позволять (36): Обстоятельства еще не [позволяют] мне [последовать] за вами .
смочь (31): Понял , что [писать] не [смогу] .
начинать (25): Я настолько безумен , что [начинаю] [интересоваться] берберами .
пытаться (25): Многое мы [пытаемся] если не [оправдать] , то объяснить периодом застоя .
успеть (20): Жаль , что не [успел] этого [сделать] .
решить (19): Синай совсем недалеко , и я [решил] [побывать] там .


In [120]:
Counter((h, i) for h, i, _ in inf_rows).most_common(10)

[(('мочь', 'быть'), 48),
 (('мочь', 'стать'), 9),
 (('мочь', 'дать'), 6),
 (('мочь', 'получить'), 5),
 (('мочь', 'существовать'), 4),
 (('мочь', 'делать'), 4),
 (('мочь', 'случиться'), 4),
 (('мочь', 'выполнять'), 3),
 (('начать', 'применять'), 3),
 (('позволять', 'получать'), 3)]

## Выводы
**Query 1: роль «который» в относительном придаточном.** Нашлось 424 примера. Почти в половине случаев (205) «который» выступает подлежащим: например, «человек, который примет решение». Ещё 124 случая относятся к `obl`, 50 — к прямому дополнению, 31 — к пассивному подлежащему. Остальные варианты (`iobj`, `obl:tmod`) встречаются редко. Получается, что чаще всего «который» обозначает того или то, что выполняет действие в придаточном, а не объект действия. При этом запрос находит только случаи, где «который» напрямую зависит от глагола, поэтому часть других относительных конструкций в результат не попадает.

**Query 2: глагол + зависимый инфинитив.** Чаще всего с инфинитивом встречается *мочь* — 302 раза, а вместе со *смочь* — ещё больше. Дальше идут *начать*, *стать*, *хотеть*, *позволять*, *начинать*, *пытаться* и другие глаголы. Самая частая пара — «мочь + быть» (48 случаев). По такому запросу можно посмотреть, какие глаголы в корпусе часто сочетаются с инфинитивом, и какие сочетания встречаются чаще всего.

Оба запроса используют именно синтаксические связи между словами, а не просто ищут слова рядом друг с другом. Дополнительные фильтры помогли убрать часть лишних примеров: в первом запросе мы оставили только придаточные с глаголом в вершине, а во втором исключили деепричастия и повторы одного и того же глагола.